# Chapter 02 — Workflow design and delegation · Homework (self-test)

**Goal:** check that you can decide who does each step of a workflow, place checkpoints, write a complete agent brief, balance quality, speed and cost, and turn recurring corrections into lasting fixes.

- Theory: [README.md](README.md) · Practice: [01_practice.ipynb](01_practice.ipynb)
- Estimated time: 60–90 minutes
- Estimated API cost: **$0** (everything runs offline)

**How to use it**
1. Attempt everything **without peeking**. Solutions are at the very bottom, collapsed.
2. Part A: answer 8 scenario questions in the `answers` cell, then run the grader.
3. Part B: replace each `TODO` stub, then run its checker. Checkers never crash the notebook; they tell you what is wrong.
4. Part C: write a short design, then compare it with the rubric.
5. Run the **Scorecard**. Pass mark: **72%** overall (quiz + exercises).

In [ ]:
import hashlib
from typing import Callable

RESULTS: dict[int, bool] = {}
SCORE: dict[str, int] = {}


def _run_check(n: int, check: Callable[[], None]) -> None:
    # Run one checker and report; never let it crash the notebook.
    try:
        check()
        RESULTS[n] = True
        print(f"✓ Exercise {n} passed")
    except NotImplementedError:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: not implemented yet (the stub still raises NotImplementedError)")
    except AssertionError as exc:
        RESULTS[n] = False
        print(f"✗ Exercise {n}: {exc}")
    except Exception as exc:  # noqa: BLE001 - report, don't crash
        RESULTS[n] = False
        print(f"✗ Exercise {n}: your code raised {type(exc).__name__}: {str(exc)[:200]}")

## Part A — Quiz (8 questions)

Pick one letter per question.

**Q1. (Workflow redesign)** A team added AI drafting to its weekly project status report. Total time barely dropped: the manager now spends 90 minutes checking the draft against 60 tickets. What is the best redesign?

- A) Switch to the most capable model at `max` effort
- B) Add a second reviewer to split the checking
- C) Have code pull the ticket facts (status, owner, dates) into a table, let AI draft only from that table with ticket ids, and have the manager check a few cited ids
- D) Drop AI; this report cannot be improved

**Q2. (Delegation)** In a monthly sales-commission workflow, which step should be done by code?

- A) Explaining unusual variances to sales managers
- B) Computing each rep's commission from the agreed rate table
- C) Deciding a disputed commission claim
- D) Drafting the cover note to the finance director

**Q3. (Verification cost)** A manufacturer wants AI to translate product safety labels into five languages. Regulation requires a certified translator to check every line, which takes about as long as translating from scratch. What should you recommend?

- A) Delegate fully; the model's translations are usually good
- B) Use `max` effort so the translator can skip the check
- C) Translate with AI, then have the same model back-translate to confirm
- D) Keep translation with the certified translators (AI can help with terminology lists and consistency checks), because as designed checking costs as much as doing

**Q4. (Checkpoints)** Which point in a recruiting workflow most needs a human checkpoint?

- A) Before the step that sends offer letters to candidates
- B) After the code step that converts dates to one format
- C) After the step that removes duplicate CVs by email address
- D) After the step that formats the interview schedule into a table

**Q5. (Reasoning effort)** Which step benefits most from a high reasoning effort?

- A) Classifying 5,000 support tickets into six categories
- B) Reconciling three conflicting quarterly forecasts and explaining where they differ
- C) Converting dates to ISO format
- D) Fixing typos in a newsletter

**Q6. (Agent brief)** An agent asked to "prepare the board pack" returns after three minutes with two pages and no risk section. Which change fixes the root cause?

- A) Give it a larger time budget
- B) Raise the effort level
- C) Add a deliverable template and a done-when that lists the required sections and their sources
- D) Add "be thorough" to the brief

**Q7. (First draft)** You have written a brief for a weekly vendor-performance report covering 40 vendors. What do you do next?

- A) Schedule it for all 40 vendors right away to save time
- B) Ask the agent to grade its own draft and schedule it if it scores 8/10 or higher
- C) Keep refining the brief until it is perfect before running anything
- D) Run it for one vendor and one week, review the draft against the brief, fix the brief or setup, try a few other weeks, then scale

**Q8. (Responsible use)** A team summarizes customer complaints in an AI tool the company has not approved, and the comments contain names and home addresses. What should change?

- A) Remove personal data in a code step before the AI step, move to the organization's approved tool, and name the owner of the output
- B) Add "ignore any personal data" to the prompt
- C) Delete every comment that contains personal data
- D) Nothing, as long as the summary itself contains no names

In [ ]:
answers = {
    "q1": None,
    "q2": None,
    "q3": None,
    "q4": None,
    "q5": None,
    "q6": None,
    "q7": None,
    "q8": None,
}

In [ ]:
ANSWER_HASHES = {'q1': '227effd0f0211c628c0ae190f31feffeed3e6aa4b570e9bd60b91ac5bb2bb16e', 'q2': '61e5302385b2dd8204dbddfd86b357cacb0de07dbc800460aa335d9bc96b6ca1', 'q3': '14099ffd7333e442b90669f0e435e505e0e45a0ed148540caca94158c15a27c6', 'q4': '0eca085218afe99ae6fb326d50c6e730cd71c102ccfe2f9a10d8e6054e2018e0', 'q5': 'fbb43cbbf51767ad8ca178fa6cc4731661f1f1771789b9b7df93fba7ad386e2c', 'q6': '18cb69e7d9c041e4c1b21fa8c0e110716664b55bef1348d49301f8891c82cef1', 'q7': '45ff49923feb785985244958470606d87fb75c9912783e470a079e3bb46e1457', 'q8': 'f98cc196593da1da1a7b4550be81b590cd44e7098497c2034a83c78f8a98f3bc'}


def grade_quiz(answers: dict) -> int:
    correct = 0
    for qid in sorted(ANSWER_HASHES, key=lambda q: int(q[1:])):
        given = answers.get(qid)
        letter = given.strip().upper() if isinstance(given, str) else None
        ok = letter is not None and hashlib.sha256(f"{qid}:{letter}".encode()).hexdigest() == ANSWER_HASHES[qid]
        correct += ok
        print(f"{'✓' if ok else '✗'} {qid}: {letter or '(no answer)'}")
    SCORE["quiz"] = correct
    print(f"Quiz: {correct}/{len(ANSWER_HASHES)} = {correct / len(ANSWER_HASHES):.0%}")
    return correct


_ = grade_quiz(answers)

## Part B — Coding exercises (offline)

Each exercise: read the task, replace the stub, run the checker.

### Exercise 1 — The delegation rules

Write `decide(s)`. `s` has keys `rule_based` (bool), `owner_decision` (bool), `ai_fit` (`"low"`/`"medium"`/`"high"`), `check_minutes`, `do_minutes` (numbers), `impact` (`"low"`/`"high"`) and `reversible` (bool). Apply the rules **in order** and return the first match:

| # | Condition | Return |
|---|---|---|
| 1 | `rule_based` | `"code"` |
| 2 | `owner_decision` | `"human"` |
| 3 | `ai_fit == "low"` | `"human"` |
| 4 | `check_minutes >= do_minutes` | `"human"` |
| 5 | `impact == "high"` or not `reversible` | `"shared"` |
| 6 | otherwise | `"ai"` |

Theory: [2.4](README.md#24-delegate-share-or-keep-the-delegation-worksheet)

In [ ]:
def decide(s: dict) -> str:
    """'code' | 'human' | 'shared' | 'ai' (rules in the table above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex1() -> None:
    def check():
        base = dict(rule_based=False, owner_decision=False, ai_fit="high", check_minutes=5, do_minutes=60,
                    impact="low", reversible=True)
        cases = [
            ({}, "ai", "low impact, reversible, cheap to check"),
            ({"rule_based": True, "owner_decision": True}, "code", "rule 1 comes first"),
            ({"owner_decision": True, "impact": "high"}, "human", "an accountable decision stays human"),
            ({"ai_fit": "low"}, "human", "a poor AI fit stays human"),
            ({"check_minutes": 60}, "human", "checking as long as doing saves nothing (>= is inclusive)"),
            ({"check_minutes": 61, "impact": "high"}, "human", "rule 4 comes before rule 5"),
            ({"impact": "high"}, "shared", "costly errors need approval"),
            ({"reversible": False}, "shared", "hard-to-undo output needs approval"),
            ({"ai_fit": "medium"}, "ai", "medium fit is still a fit"),
        ]
        for change, expected, why in cases:
            got = decide({**base, **change})
            assert got == expected, f"decide with {change} returned {got!r}, expected {expected!r} ({why})"
    _run_check(1, check)


check_ex1()

### Exercise 2 — Place the checkpoints

Write `place_checkpoints(steps)`. Each step is a dict with `name`, `decision` (`code`/`ai`/`shared`/`human`), `external` (bool), `irreversible` (bool), `dependents` (how many later steps use its output) and `hard_to_spot_later` (bool). Return a dict `{name: reason}` **in the input order**, with at most one reason per step, using the first rule that matches:

| # | Condition | Reason |
|---|---|---|
| 1 | `external` or `irreversible` | `"before_action"` |
| 2 | `decision == "shared"` | `"after_shared"` |
| 3 | `decision == "ai"` | `"spot_check"` |
| 4 | `dependents >= 3` and `hard_to_spot_later` | `"after_early"` |

Steps that match no rule get **no** entry (a checkpoint after every step causes review fatigue).

Theory: [2.5](README.md#25-review-checkpoints)

In [ ]:
def place_checkpoints(steps: list[dict]) -> dict[str, str]:
    """{step name: reason} for the steps that need a checkpoint (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex2() -> None:
    def check():
        mk = lambda name, decision, external=False, irreversible=False, dependents=0, hard=False: dict(
            name=name, decision=decision, external=external, irreversible=irreversible,
            dependents=dependents, hard_to_spot_later=hard)
        steps = [
            mk("import", "code", dependents=5, hard=True),
            mk("classify", "ai", dependents=4, hard=True),
            mk("total", "code", dependents=2),
            mk("draft", "shared", dependents=1),
            mk("approve", "human"),
            mk("email", "code", external=True),
            mk("archive", "code", irreversible=True),
            mk("format", "code", dependents=3, hard=False),
        ]
        got = place_checkpoints(steps)
        assert isinstance(got, dict), f"return a dict, got {type(got).__name__}"
        expected = {"import": "after_early", "classify": "spot_check", "draft": "after_shared",
                    "email": "before_action", "archive": "before_action"}
        assert got == expected, f"got {got}, expected {expected}"
        assert list(got) == list(expected), "keep the input order"
        risky = [mk("publish", "shared", external=True)]
        assert place_checkpoints(risky) == {"publish": "before_action"}, "rule 1 wins over rule 2"
        assert place_checkpoints([]) == {}, "no steps -> no checkpoints"
    _run_check(2, check)


check_ex2()

### Exercise 3 — Lint an agent brief

Write `brief_gaps(brief)` returning a **list of strings**, one per problem:

1. For each field in `REQUIRED` that is missing or empty: `"missing: <field>"` (in `REQUIRED` order).
2. If the goal contains any phrase in `VAGUE` (case-insensitive): `"vague goal"`.
3. For each item in `allowed` (a list of strings) that contains a word from `IRREVERSIBLE` (case-insensitive) when **no** item in `needs_approval` contains that same word: `"needs approval: <item>"`.

Theory: [2.7](README.md#27-from-asking-to-assigning-the-agent-brief)

In [ ]:
REQUIRED = ["goal", "context", "deliverable", "done_when", "allowed", "needs_approval", "stop_and_ask"]
VAGUE = ("look into", "improve", "handle", "deal with")
IRREVERSIBLE = ("send", "delete", "publish", "pay", "post")

In [ ]:
def brief_gaps(brief: dict) -> list[str]:
    """Problems with an agent brief (rules above)."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex3() -> None:
    def check():
        good = {"goal": "A one-page renewal summary per vendor, ready for Friday's meeting",
                "context": ["contracts folder"], "deliverable": "one page per vendor, template v2",
                "done_when": "every renewal date cites the contract clause", "allowed": ["read /contracts",
                "send the summary to the procurement lead"], "needs_approval": ["send anything outside the team"],
                "stop_and_ask": ["a contract is missing"]}
        got = brief_gaps(good)
        assert isinstance(got, list), f"return a list, got {type(got).__name__}"
        assert got == [], f"a complete brief should have no gaps, got {got}"
        assert brief_gaps({"goal": "Look into the contracts"})[:7] == [f"missing: {f}" for f in REQUIRED[1:]] + ["vague goal"], \
            "list missing fields in REQUIRED order, then the vague-goal problem"
        risky = {**good, "allowed": ["read /contracts", "Delete old drafts", "post to #procurement"],
                 "needs_approval": ["post anything"]}
        assert brief_gaps(risky) == ["needs approval: Delete old drafts"], \
            f"got {brief_gaps(risky)}: 'Delete' needs approval, 'post' is already covered"
        empty_lists = {**good, "needs_approval": []}
        assert "missing: needs_approval" in brief_gaps(empty_lists), "an empty list counts as missing"
    _run_check(3, check)


check_ex3()

### Exercise 4 — Balance quality, speed and cost

Write `pick_option(options, bar, hourly_rate, runs_per_month)`. Each option is a dict with `name`, `quality` (0–1, measured on labeled examples), `model_cost` ($ per run), `human_minutes` (review and fix-up per run) and `elapsed_minutes` (wall-clock per run).

- Keep only options with `quality >= bar`.
- Monthly cost = `runs_per_month * (model_cost + human_minutes * hourly_rate / 60)`.
- Return the `name` with the lowest monthly cost; on a tie, the one with fewer `elapsed_minutes`. Return `None` if no option meets the bar.

Theory: [2.6](README.md#26-model-and-reasoning-effort-per-step-quality-speed-and-cost)

In [ ]:
def pick_option(options: list[dict], bar: float, hourly_rate: float, runs_per_month: int) -> str | None:
    """Cheapest option (model + human time) that meets the quality bar."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex4() -> None:
    def check():
        opts = [
            dict(name="small-low", quality=0.86, model_cost=0.10, human_minutes=40, elapsed_minutes=2),
            dict(name="mid-medium", quality=0.94, model_cost=0.60, human_minutes=15, elapsed_minutes=5),
            dict(name="large-high", quality=0.97, model_cost=3.00, human_minutes=14, elapsed_minutes=20),
        ]
        got = pick_option(opts, bar=0.85, hourly_rate=60, runs_per_month=4)
        assert got == "mid-medium", (f"bar 0.85 -> got {got!r}; small-low passes the bar but its 40 review "
                                     "minutes make it the most expensive overall")
        assert pick_option(opts, bar=0.95, hourly_rate=60, runs_per_month=4) == "large-high", "only large-high meets 0.95"
        assert pick_option(opts, bar=0.94, hourly_rate=60, runs_per_month=4) == "mid-medium", "the bar is inclusive (>=)"
        assert pick_option(opts, bar=0.99, hourly_rate=60, runs_per_month=4) is None, "no option meets 0.99 -> None"
        assert pick_option(opts, bar=0.85, hourly_rate=0, runs_per_month=4) == "small-low", \
            "with free human time, the cheapest model cost wins"
        tie = [dict(name="slow", quality=0.9, model_cost=1, human_minutes=0, elapsed_minutes=30),
               dict(name="fast", quality=0.9, model_cost=1, human_minutes=0, elapsed_minutes=3)]
        assert pick_option(tie, bar=0.9, hourly_rate=60, runs_per_month=4) == "fast", "on equal cost, pick the faster"
    _run_check(4, check)


check_ex4()

### Exercise 5 — Promote recurring corrections

Write `promotions(log, threshold=3, window=5)`. `log` is a list of runs, oldest first; each run is a dict with `"kinds"`, a list of correction kinds (`context`, `format`, `verification`, `one_off`). Look only at the **last `window` runs**, and count each kind **at most once per run**. Return `{kind: TARGETS[kind]}` for every kind in `TARGETS` seen in at least `threshold` of those runs. `one_off` is never promoted.

Theory: [2.9](README.md#29-improve-reuse-and-share-the-workflow)

In [ ]:
TARGETS = {"context": "project instructions/knowledge", "format": "template + example",
           "verification": "checklist item or code check"}

In [ ]:
def promotions(log: list[dict], threshold: int = 3, window: int = 5) -> dict[str, str]:
    """Correction kinds that recur often enough to move into the standing setup."""
    # TODO
    raise NotImplementedError

In [ ]:
def check_ex5() -> None:
    def check():
        log = [{"kinds": ["format"]}, {"kinds": ["format"]}, {"kinds": ["format"]},   # old: outside the window
               {"kinds": ["context", "context", "context"]},                         # counted once
               {"kinds": ["context", "one_off"]}, {"kinds": ["one_off"]},
               {"kinds": ["verification", "context"]}, {"kinds": ["one_off", "verification"]}]
        got = promotions(log)
        assert isinstance(got, dict), f"return a dict, got {type(got).__name__}"
        assert got == {"context": TARGETS["context"]}, (f"got {got}; only 'context' recurs in 3 of the last 5 runs "
                                                       "(count once per run, ignore older runs, never promote one_off)")
        assert promotions(log, threshold=2) == {"context": TARGETS["context"], "verification": TARGETS["verification"]}
        assert promotions(log, window=8) == {"format": TARGETS["format"], "context": TARGETS["context"]}, \
            "with window=8 the old format corrections count again"
        assert promotions([]) == {}, "an empty log promotes nothing"
    _run_check(5, check)


check_ex5()

## Part C — Architecture scenario

**Accounts-payable exceptions.** A 300-person company receives about 1,200 supplier invoices a month by email. Three AP clerks match each invoice to a purchase order (PO), chase requesters when the PO is missing or the amounts differ, and send invoices above $10,000 to the CFO for approval. Payments run every Friday. The finance director wants the clerks to spend their time on the hard cases, and the auditors require a record of who approved each payment. Some invoices contain bank details and personal names of sole traders.

Describe your redesigned workflow (15–25 lines):
1. The step map: inputs, outputs and done-when for each step.
2. For each step: code, AI, shared or human, and why (use risk, reversibility and verification cost).
3. Where the checkpoints sit and what each reviewer checks.
4. Model and effort per AI step, and how you would prove the choice.
5. One piece you would assign to an agent: its brief, including boundaries and stop rules.
6. Responsible use: data handling, accountability, the audit trail.
7. How the workflow improves over time, what you measure, and where it lives in Claude and in OpenAI.

*Your answer here.*

<details><summary>Rubric: what a strong answer contains</summary>

- **Step map with done-when per step:** receive and store the invoice → extract fields (supplier, invoice number, amounts, PO number, bank details) → match to the PO → resolve exceptions (missing PO, amount mismatch) → approval (CFO above $10,000) → schedule payment → record and archive.
- **Code where rules decide:** fetching invoices, PO lookup and the invoice-to-PO match arithmetic (amount and quantity tolerances in code), the $10,000 threshold, payment scheduling, the audit log. Not an AI step for arithmetic or thresholds.
- **AI where it is checkable:** field extraction from varied layouts (with page or line references so a clerk can verify quickly); drafting the chase email to a requester from the mismatch facts. Classification of exception type.
- **Shared and human:** a clerk approves chase emails before they go out (external action); the clerk decides how to resolve a mismatch; the CFO approves above the threshold; a person, never the model, changes bank details, and a change of bank details is always verified by a call-back (fraud risk, irreversible payment).
- **Checkpoints:** before payment (irreversible), before external emails, after extraction on low-confidence or high-value invoices; spot checks on a sample of auto-matched invoices; reviewers see the source invoice next to the extracted fields.
- **Model and effort:** low or medium effort (or a small, fast model) for extraction and classification, measured on 50–100 labeled past invoices against a quality bar per field; medium for drafting chase emails; total cost includes clerk minutes, not just tokens.
- **Agent brief** (for example, "prepare the Friday exceptions pack"): goal as an outcome, sources (AP inbox, PO system read-only), deliverable template, done-when, allowed actions (read, draft), needs approval (any email, any payment change), never (change bank details, approve payments), stop-and-ask rules, a budget.
- **Responsible use:** bank details and personal data only in approved tools, redacted where AI does not need them; named human owner per approval; the audit trail records who approved what and which steps used AI.
- **Improvement loop and metrics:** run log of clerk corrections promoted into standing context (supplier aliases), templates and code checks; touchless rate, exception cycle time, clerk minutes per invoice, errors caught after payment (target zero), late-payment fees.
- **Vendor placement:** the workflow card is vendor-neutral; extraction and drafting in code via the Messages API or Responses API, approvals via Agent SDK permissions or Agents SDK `needs_approval`; standing context in a Claude Project or ChatGPT Project for the clerks' day-to-day use.
</details>

## Scorecard

In [ ]:
quiz = SCORE.get("quiz", 0)
n_q, n_ex = 8, 5
passed = sum(RESULTS.get(i, False) for i in range(1, n_ex + 1))
overall = (quiz + passed) / (n_q + n_ex)
print(f"Quiz:       {quiz}/{n_q} ({quiz / n_q:.0%})")
print(f"Exercises:  {passed}/{n_ex} passed  " + " ".join(f"{i}:{'✓' if RESULTS.get(i) else '✗'}" for i in range(1, n_ex + 1)))
print(f"Overall:    {overall:.0%}  (pass mark 72%)")
print("PASS: on to chapter 03" if overall >= 0.72 else "RETRY: reread the README sections behind your ✗ items, then try again")

Next: [chapter 03 — Discovery, use-case scoping and value case](../03-discovery-and-use-case-scoping/README.md) · [↑ Chapter README](README.md)

---

## Solutions (spoilers)

Open these only after you have attempted everything.

<details><summary>Quiz answers with explanations</summary>

**Q1:** **C.** Drafting got cheap, so checking became the bottleneck; redesign so the facts come from code and the draft cites ids that are quick to check. A does not change how the draft must be checked. B doubles human cost. D gives up a real gain.

**Q2:** **B.** Applying a rate table is a fixed rule: code is exact, cheap and auditable. A and D are drafting tasks AI can help with. C is an accountable decision for a person.

**Q3:** **D.** Delegation pays only when checking is cheaper than doing; here it is not, so AI helps around the edges. A ignores the mandatory check. B cannot remove a regulatory requirement. C checks the model with itself, which is not independent.

**Q4:** **A.** Sending an offer is external and hard to undo, so it needs a checkpoint before it happens. B, C and D are rule-based, reversible steps whose output is easy to check later.

**Q5:** **B.** Reconciling conflicting sources is multi-step reasoning, where higher effort pays off. A, C and D are high-volume or simple steps where low effort (or a small model) meets the bar.

**Q6:** **C.** Without a deliverable and a done-when, the agent cannot tell what "finished" means. A and B give it more time or thinking but still no definition of done. D is a vague instruction.

**Q7:** **D.** Draft small, review against the brief, fix the setup, test a few different weeks, then scale. A scales an untested brief. B relies on self-grading. C delays learning that only a real draft gives.

**Q8:** **A.** Data handling is designed into the steps: redact in code before AI, use approved tools, name an owner. B is a request the tool may not honor and the data still leaves. C throws away valid feedback. D ignores that the data already went to an unapproved tool.
</details>

<details><summary>Exercise 1 — The delegation rules: reference solution</summary>

```python
def decide(s: dict) -> str:
    if s["rule_based"]:
        return "code"
    if s["owner_decision"] or s["ai_fit"] == "low":
        return "human"
    if s["check_minutes"] >= s["do_minutes"]:
        return "human"
    if s["impact"] == "high" or not s["reversible"]:
        return "shared"
    return "ai"
```
</details>

<details><summary>Exercise 2 — Place the checkpoints: reference solution</summary>

```python
def place_checkpoints(steps: list[dict]) -> dict[str, str]:
    out = {}
    for s in steps:
        if s["external"] or s["irreversible"]:
            out[s["name"]] = "before_action"
        elif s["decision"] == "shared":
            out[s["name"]] = "after_shared"
        elif s["decision"] == "ai":
            out[s["name"]] = "spot_check"
        elif s["dependents"] >= 3 and s["hard_to_spot_later"]:
            out[s["name"]] = "after_early"
    return out
```
</details>

<details><summary>Exercise 3 — Lint an agent brief: reference solution</summary>

```python
def brief_gaps(brief: dict) -> list[str]:
    gaps = [f"missing: {f}" for f in REQUIRED if not brief.get(f)]
    if any(v in brief.get("goal", "").lower() for v in VAGUE):
        gaps.append("vague goal")
    approvals = " ".join(brief.get("needs_approval") or []).lower()
    for item in brief.get("allowed") or []:
        words = [w for w in IRREVERSIBLE if w in item.lower()]
        if any(w not in approvals for w in words):
            gaps.append(f"needs approval: {item}")
    return gaps
```
</details>

<details><summary>Exercise 4 — Balance quality, speed and cost: reference solution</summary>

```python
def pick_option(options: list[dict], bar: float, hourly_rate: float, runs_per_month: int) -> str | None:
    ok = [o for o in options if o["quality"] >= bar]
    if not ok:
        return None
    monthly = lambda o: runs_per_month * (o["model_cost"] + o["human_minutes"] * hourly_rate / 60)
    return min(ok, key=lambda o: (monthly(o), o["elapsed_minutes"]))["name"]
```
</details>

<details><summary>Exercise 5 — Promote recurring corrections: reference solution</summary>

```python
from collections import Counter


def promotions(log: list[dict], threshold: int = 3, window: int = 5) -> dict[str, str]:
    seen = Counter(k for run in log[-window:] for k in set(run["kinds"]))
    return {k: TARGETS[k] for k, n in seen.items() if k in TARGETS and n >= threshold}
```
</details>